# 12 · Embedding Geometry and Distance

This notebook quantifies the geometry seen in Notebook 10 and the separability seen in Notebook 11.

We work with the same 5-gap descriptor vectors and ask:

1. how far apart are the ensemble centroids?
2. how much do local neighborhoods mix across ensembles?
3. how similar are the 2D density patterns in PCA space?

The main comparison is:

- zeta vs GUE
- zeta vs Poisson
- GUE vs Poisson

## Why this matters

If zeta and GUE are genuinely close in descriptor space, we should see:

- smaller centroid distance
- more local neighborhood mixing
- more similar coarse density patterns

than for comparisons involving Poisson.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Data sources and 5-gap descriptors

In [ ]:
# Zeta gaps
N = 700
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# Finite GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=40, bulk_fraction=0.5, rng=rng)

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_features(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r)), float(np.std(r))

def descriptor_vector(window):
    ratio_mean, ratio_std = ratio_features(window)
    return np.array([
        window[0], window[1], window[2], window[3], window[4],
        unevenness(window),
        reversal_symmetry_score(window),
        window_entropy(window),
        ratio_mean,
        ratio_std,
    ], dtype=float)

def build_descriptors(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    desc = np.array([descriptor_vector(w) for w in windows_n])
    return windows_n, desc

zeta_w, zeta_desc = build_descriptors(zeta_gaps, k=5)
poisson_w, poisson_desc = build_descriptors(poisson_gaps, k=5)
gue_w, gue_desc = build_descriptors(gue_gaps, k=5)

zeta_desc.shape, poisson_desc.shape, gue_desc.shape

## Standardize and build PCA embedding

In [ ]:
all_desc = np.vstack([zeta_desc, poisson_desc, gue_desc])

mean = all_desc.mean(axis=0)
std = all_desc.std(axis=0)
std[std == 0] = 1.0

zeta_std = (zeta_desc - mean) / std
poisson_std = (poisson_desc - mean) / std
gue_std = (gue_desc - mean) / std

X = np.vstack([zeta_std, poisson_std, gue_std])
Xc = X - X.mean(axis=0)

cov = np.cov(Xc, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

components = eigvecs[:, :2]
embedding = Xc @ components

n_zeta = len(zeta_std)
n_poisson = len(poisson_std)
n_gue = len(gue_std)

zeta_emb = embedding[:n_zeta]
poisson_emb = embedding[n_zeta:n_zeta + n_poisson]
gue_emb = embedding[n_zeta + n_poisson:]

explained = eigvals[:2] / eigvals.sum()
explained

## PCA scatter overview

In [ ]:
plt.figure(figsize=(8.8, 6.2))
plt.scatter(zeta_emb[:, 0], zeta_emb[:, 1], s=10, alpha=0.25, label="zeta")
plt.scatter(poisson_emb[:, 0], poisson_emb[:, 1], s=10, alpha=0.25, label="Poisson")
plt.scatter(gue_emb[:, 0], gue_emb[:, 1], s=10, alpha=0.25, label="GUE")
plt.xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({explained[1]*100:.1f}% var)")
plt.title("PCA embedding overview")
plt.legend()
plt.show()

## Centroid distances

In [ ]:
centroids = {
    "zeta": zeta_emb.mean(axis=0),
    "Poisson": poisson_emb.mean(axis=0),
    "GUE": gue_emb.mean(axis=0),
}

centroid_distances = {
    "zeta_GUE": float(np.linalg.norm(centroids["zeta"] - centroids["GUE"])),
    "zeta_Poisson": float(np.linalg.norm(centroids["zeta"] - centroids["Poisson"])),
    "GUE_Poisson": float(np.linalg.norm(centroids["GUE"] - centroids["Poisson"])),
}
centroid_distances

In [ ]:
labels = list(centroid_distances.keys())
values = [centroid_distances[k] for k in labels]

plt.figure(figsize=(7.8, 4.8))
plt.bar(labels, values)
plt.ylabel("centroid distance in PCA space")
plt.title("Centroid-distance comparison")
plt.show()

## Local neighborhood mixing

For each point, look at its k nearest neighbors in PCA space and measure the fraction belonging to each ensemble.

If zeta and GUE are locally similar, we expect:
- zeta points to have many GUE neighbors
- GUE points to have many zeta neighbors
- Poisson neighborhoods to be more self-contained

In [ ]:
all_emb = np.vstack([zeta_emb, poisson_emb, gue_emb])
all_labels = np.array(
    ["zeta"] * len(zeta_emb) +
    ["Poisson"] * len(poisson_emb) +
    ["GUE"] * len(gue_emb)
)

def knn_label_fractions(X, labels, k=15):
    n = len(X)
    unique_labels = np.unique(labels)
    out = {lab: [] for lab in unique_labels}

    for i in range(n):
        d = np.linalg.norm(X - X[i], axis=1)
        nn = np.argsort(d)[1:k+1]
        nn_labels = labels[nn]
        for lab in unique_labels:
            out[lab].append(np.mean(nn_labels == lab))

    return {lab: np.array(vals) for lab, vals in out.items()}

fractions = knn_label_fractions(all_emb, all_labels, k=15)
fractions.keys()

In [ ]:
def summarize_neighbor_mix(target_label):
    mask = all_labels == target_label
    return {
        neighbor_label: float(fractions[neighbor_label][mask].mean())
        for neighbor_label in fractions
    }

neighbor_mix_summary = {
    "zeta_points": summarize_neighbor_mix("zeta"),
    "Poisson_points": summarize_neighbor_mix("Poisson"),
    "GUE_points": summarize_neighbor_mix("GUE"),
}
neighbor_mix_summary

## Cross-ensemble neighbor fraction plot

In [ ]:
groups = ["zeta_points", "Poisson_points", "GUE_points"]
neighbor_labels = ["zeta", "Poisson", "GUE"]

x = np.arange(len(groups))
width = 0.25

plt.figure(figsize=(8.8, 5.0))
for j, lab in enumerate(neighbor_labels):
    vals = [neighbor_mix_summary[g][lab] for g in groups]
    plt.bar(x + (j-1)*width, vals, width, label=f"{lab} neighbors")

plt.xticks(x, groups)
plt.ylim(0, 1)
plt.ylabel("mean neighbor fraction")
plt.title("Local neighborhood mixing in PCA space")
plt.legend()
plt.show()

## Coarse 2D density histograms

To compare ensemble shapes quantitatively, we bin PCA space into a 2D grid and compute normalized occupancy maps.

In [ ]:
xmin = min(zeta_emb[:,0].min(), poisson_emb[:,0].min(), gue_emb[:,0].min())
xmax = max(zeta_emb[:,0].max(), poisson_emb[:,0].max(), gue_emb[:,0].max())
ymin = min(zeta_emb[:,1].min(), poisson_emb[:,1].min(), gue_emb[:,1].min())
ymax = max(zeta_emb[:,1].max(), poisson_emb[:,1].max(), gue_emb[:,1].max())

bins_x = np.linspace(xmin, xmax, 36)
bins_y = np.linspace(ymin, ymax, 36)

def density_grid(X):
    H, _, _ = np.histogram2d(X[:,0], X[:,1], bins=[bins_x, bins_y], density=True)
    return H

zeta_H = density_grid(zeta_emb)
poisson_H = density_grid(poisson_emb)
gue_H = density_grid(gue_emb)

zeta_H.shape

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, H, title in zip(
    axes,
    [zeta_H, poisson_H, gue_H],
    ["zeta density", "Poisson density", "GUE density"]
):
    ax.imshow(
        H.T,
        origin="lower",
        aspect="auto",
        extent=[xmin, xmax, ymin, ymax]
    )
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()

## Density-map distance

We compare the 2D histograms with a simple Euclidean distance between flattened density grids.
This is crude but useful as a first overlap measure.

In [ ]:
def grid_distance(H1, H2):
    return float(np.linalg.norm(H1.ravel() - H2.ravel()))

grid_distance_summary = {
    "zeta_GUE": grid_distance(zeta_H, gue_H),
    "zeta_Poisson": grid_distance(zeta_H, poisson_H),
    "GUE_Poisson": grid_distance(gue_H, poisson_H),
}
grid_distance_summary

In [ ]:
labels = list(grid_distance_summary.keys())
values = [grid_distance_summary[k] for k in labels]

plt.figure(figsize=(7.8, 4.8))
plt.bar(labels, values)
plt.ylabel("density-grid distance")
plt.title("Coarse density-pattern distance")
plt.show()

## Nearest-centroid assignment regions

As another simple geometric diagnostic, assign each embedded point to the nearest ensemble centroid and compare with the true label.

In [ ]:
centroid_array = np.vstack([centroids["zeta"], centroids["Poisson"], centroids["GUE"]])
centroid_names = np.array(["zeta", "Poisson", "GUE"])

dists = np.linalg.norm(all_emb[:, None, :] - centroid_array[None, :, :], axis=2)
assigned = centroid_names[np.argmin(dists, axis=1)]

assignment_table = {}
for true_lab in ["zeta", "Poisson", "GUE"]:
    mask = all_labels == true_lab
    counts = {pred_lab: int(np.sum(assigned[mask] == pred_lab)) for pred_lab in centroid_names}
    assignment_table[true_lab] = counts

assignment_table

## Summary

In [ ]:
summary = {
    "centroid_distances": centroid_distances,
    "neighbor_mix_summary": neighbor_mix_summary,
    "grid_distance_summary": grid_distance_summary,
    "assignment_table": assignment_table,
}
summary

## Notes

- Smaller zeta–GUE centroid and density-grid distances than zeta–Poisson distances support the interpretation that zeta and GUE occupy similar regions of descriptor space.
- Local neighborhood mixing is often more informative than centroid distance alone, because two clouds can have nearby means but different local structure.
- The 2D density-grid comparison is intentionally coarse; later versions could use kernel density estimates or Wasserstein distances.

## Next directions

A natural next notebook could do one of these:

1. bootstrap centroid and grid-distance uncertainty  
2. compute Wasserstein-style distances between the embedding clouds  
3. add conditional embedding geometry for GUE and Poisson, not just zeta  
4. compare several descriptor sets and several window lengths